# Transcript Sampling Audit

## tl;dr

- Sampled **200 one-minute sections** from **40 videos** spanning **2021-04-19 to 2026-08-05**.
- Flagged **37 sections (18.5%)** for manual listening.
- Extracted **347 proper-name candidates** and **11 possible spelling/alias mismatches**.
- These are screening results. YouTube captions are a comparison source, not ground truth.


## Context & Methods

The audit deterministically selects one video from each of 40 chronological buckets among completed videos that have both native Whisper segments and YouTube captions. Each selected video contributes windows at approximately 5%, 27%, 50%, 73%, and 95% of its duration.

### Key Assumptions

- Low agreement identifies sections worth listening to; it does not establish which transcript is correct.
- Timing drift, censorship tokens, music labels, and caption omissions can lower agreement without a Whisper error.
- Proper-name extraction is intentionally broad. Low-confidence capitalized phrases remain candidates until manually checked.


In [1]:
from pathlib import Path
import csv, json, statistics

AUDIT_DIR = Path('output/transcript-audit-2026-08-11')

def read_csv(name):
    with (AUDIT_DIR / name).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

summary = json.loads((AUDIT_DIR / 'summary.json').read_text(encoding='utf-8'))
samples = read_csv('sampled_sections.csv')
flags = read_csv('flagged_sections.csv')
names = read_csv('name_candidates.csv')
name_leads = read_csv('possible_name_mistranscriptions.csv')
issue_counts = read_csv('issue_counts.csv')
summary

{'sampled_videos': 40,
 'sampled_sections': 200,
 'flagged_sections': 37,
 'flagged_share': 0.185,
 'name_rows': 347,
 'canonical_name_rows': 20,
 'unverified_name_rows': 327,
 'possible_name_mistranscriptions': 11,
 'sample_positions': [0.05, 0.27, 0.5, 0.73, 0.95],
 'issue_counts': {'caption disagreement': 12,
  'lower caption agreement': 17,
  'Whisper missed captioned speech': 5,
  'strong caption disagreement': 1,
  'known name variant': 2,
  'heavy repetition': 1},
 'minimum_uploaded_date': '2021-04-19',
 'maximum_uploaded_date': '2026-08-05',
 'method_note': 'Flags are screening signals. YouTube captions are a comparison source, not ground truth.'}

## Data

The sampled row grain is one 60-second window per video-position pair. Exact SQL is saved in `sampling_query.sql`; full rows are retained in `sampled_sections.csv`.

In [2]:
print('Sampled rows:', len(samples))
print('Distinct videos:', len({row['youtube_id'] for row in samples}))
print('Positions:', sorted({row['sample_position'] for row in samples}))
print('Date range:', min(row['uploaded_date'] for row in samples), 'to', max(row['uploaded_date'] for row in samples))

Sampled rows: 200
Distinct videos: 40
Positions: ['early', 'first_quarter', 'late', 'middle', 'third_quarter']
Date range: 2021-04-19 to 2026-08-05


## Results

### Manual-review volume by screening signal

In [3]:
for row in issue_counts:
    print(f"{row['issue_type']:<34} {int(row['flagged_sections']):>3} sections ({float(row['share']):.1%})")

lower caption agreement             17 sections (8.5%)
caption disagreement                12 sections (6.0%)
Whisper missed captioned speech      5 sections (2.5%)
known name variant                   2 sections (1.0%)
strong caption disagreement          1 sections (0.5%)
heavy repetition                     1 sections (0.5%)


### Highest-priority timestamp checks

In [4]:
for row in flags[:15]:
    print(f"[{row['review_priority']}] {row['uploaded_date']} {row['timestamp']} — {row['video_title']}")
    print('Reason:', row['issues'])
    print('Possible differences:', row['possible_word_or_name_differences'] or '(alignment unavailable)')
    print('URL:', row['youtube_url'])
    print()

[7] 2025-04-19 01:40:22 — HasanAbi April 18, 2025 – Live w/ KNEEPCAP at Coachella
Reason: heavy repetition; caption disagreement
Possible differences: stream → streamed; They're → They are; two → 2
URL: https://www.youtube.com/watch?v=2zzyqxWzQuc&t=6022s

[5] 2021-07-27 00:10:20 — 1/2 HasanAbi July 26, 2021 – Hachubby Shake Shack Drama, DaBaby dodges shoe, DELTA Variant spreads
Reason: Whisper missed captioned speech
Possible differences: (alignment unavailable)
URL: https://www.youtube.com/watch?v=In2iYbpNHp8&t=620s

[5] 2021-08-06 00:06:17 — 1/2 HasanAbi August 5, 2021 – Crowder gets surgury, Fauci on Delta, Bill Gates on Epstein, ICU Vlogs
Reason: Whisper missed captioned speech
Possible differences: (alignment unavailable)
URL: https://www.youtube.com/watch?v=3ha9GYJvlrQ&t=377s

[5] 2021-12-13 00:11:06 — 1/2 HasanAbi November 18, 2021 – Hasan gets a Haircut, Ahmaud Arbery & Rittenhouse Trials, CHANNEL 5
Reason: Whisper missed captioned speech
Possible differences: (alignment unavai

### Possible name spellings and aliases to verify

In [5]:
for row in name_leads:
    print(f"{row['observed_form']} -> {row['possible_correction']} ({row['similarity']})")
    print(f"  {row['timestamp']} | {row['video_title']}")
    print(f"  {row['youtube_url']}")

Abdul El Sayed -> Abdul El-Sayed (1.0)
  06:09:28 | HasanAbi April 3, 2026 –
  https://www.youtube.com/watch?v=aPgeOA4mWqg&t=22168s
Austin -> AustinShow (1.0)
  01:30:27 | [15/11/2024] –🏀 OTK Gameday Basketball Event ft. HasanAbi
  https://www.youtube.com/watch?v=MnCX6pnvxcc&t=5427s
Emma Vigland -> Emma Vigeland (0.96)
  06:09:28 | HasanAbi April 3, 2026 –
  https://www.youtube.com/watch?v=aPgeOA4mWqg&t=22168s
Bradley Martin -> Bradley Martyn (0.929)
  00:20:34 | HasanAbi March 7, 2025 – Noah Kulwin (BlowBack) is still here, 🎮Marvel Rivals🎮
  https://www.youtube.com/watch?v=i8YmtStCJJo&t=1234s
Carl Marx -> Karl Marx (0.889)
  03:34:32 | 2/2 HasanAbi September 13, 2021 – AOC at the MET GALA, Making fun of Preppers, OKBUDDY
  https://www.youtube.com/watch?v=MmJgjK3Zsy4&t=12872s
New Yorker -> New York (0.889)
  05:36:15 | HasanAbi April 19, 2026 –
  https://www.youtube.com/watch?v=FyI1n2mp8L4&t=20175s
Aiden Ross -> Adin Ross (0.842)
  00:06:32 | 2/2 HasanAbi May 26, 2021 – LSF, New Twitch

### Name-candidate confidence profile

In [6]:
from collections import Counter
print(Counter(row['confidence'] for row in names))
print('Top recurring candidates:')
for row in names[:20]:
    print(row['sample_occurrences'], row['confidence'], row['canonical_or_candidate'], row['first_timestamp'])

Counter({'low': 262, 'medium': 55, 'high': 30})
Top recurring candidates:
9 high Donald Trump 03:11:18
9 high New York 00:51:57
8 high United States 01:43:20
5 medium Democratic Party 03:09:05
5 medium Fox News 02:46:13
5 medium Joe Biden 01:18:30
4 high Bernie Sanders 03:46:14
4 high Joe Rogan 04:39:12
3 medium Hillary Clinton 03:32:17
3 medium Jeffrey Epstein 03:32:17
2 medium Al Turkey 00:25:45
2 high Andrew Yang 00:02:37
2 medium Anthony Davis 00:20:36
2 medium Are Charlie Kirk 01:58:02
2 high AustinShow 01:30:27
2 medium BBC Persian 04:17:31
2 medium Bear County 05:51:08
2 medium Ben Hamas Rhodes 04:18:23
2 high Ben Rhodes 04:18:23
2 medium Bernard Sanders 00:26:40


## Takeaways

- Begin manual review with the highest-priority disagreement and repetition rows, then work down the complete flagged-section CSV.
- Treat fuzzy name suggestions as hypotheses. Several are likely genuine (`Emma Vigland`, `Bradley Martin`, `Carl Marx`, `Aiden Ross`), while generic phrase matches can be false positives.
- Use verified corrections to expand the decoder prompt and contextual replacement list; do not automatically rewrite the archive from this screening output.
